# Aral Sea Model

Use this to generate meteorological forcing for the Aral Sea model. Run only once: it writes outputs to the save location defined in [`config_aral.yaml`](../config_aral.yaml) and uses the model list and shapefile from that same config by default.



## Setup imports and configuration

- Sets `PROJECT_ROOT` and ensures `src` is on `sys.path`.
- Imports plotting and YAML libraries and the forcing generator functions from `src.forcing`.
- Loads `config_aral.yaml` into `config` and defines `resolve_project_path()`.

Purpose: prepare the runtime environment and configuration used by later cells.

In [1]:
# setup imports
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import importlib
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



from src.forcing import generate_lumped_CMIP_historical_forcing, generate_lumped_CMIP_future_forcing, generate_lumped_ERA5_forcing



# load the project configuration
config_path = PROJECT_ROOT / "config_aral.yaml"
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
def resolve_project_path(config_path_value):
    return PROJECT_ROOT / config_path_value

    
config_path, config["project"]["title"]





/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


(PosixPath('/home/avandervee3/aral_sea_full_project/config_aral.yaml'),
 'Aral Sea Level Modelling')

## Confirm model list

- Reads `list_models = config['forcing']['models']` and prints it.

Purpose: verify which CMIP models are configured for processing.

In [3]:
list_models = config["forcing"]["models"]
list_models


['MIROC6', 'CanESM5', 'FGOALS-g3', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-LM']

## Create forcing output directories

- Creates `ARAL_FORCING_FOLDER` and subfolders `ERA5`, `CMIP6_historical`, and `CMIP6_future` if they don't exist.

Purpose: ensure folders are ready to receive generated forcing files.

In [ ]:
ARAL_FORCING_FOLDER = resolve_project_path(config["paths"]["forcing_dir"]) / "aral_sea"
ARAL_FORCING_FOLDER.mkdir(parents=True, exist_ok=True)
ARAL_FORCING_ERA5_FOLDER = ARAL_FORCING_FOLDER / "ERA5"
ARAL_FORCING_ERA5_FOLDER.mkdir(parents=True, exist_ok=True)
ARAL_FORCING_CMIP6_HIST_FOLDER = ARAL_FORCING_FOLDER / "CMIP6_historical"
ARAL_FORCING_CMIP6_HIST_FOLDER.mkdir(parents=True, exist_ok=True)
ARAL_FORCING_CMIP6_FUTURE_FOLDER = ARAL_FORCING_FOLDER / "CMIP6_future"
ARAL_FORCING_CMIP6_FUTURE_FOLDER.mkdir(parents=True, exist_ok=True)


## Resolve shapefile path

- Sets `shapefile` from `config['aral_sea_experiment']['shapefile_path']`.

Purpose: provide the study-area geometry used to generate forcing data.

In [ ]:
shapefile = resolve_project_path(config["aral_sea_experiment"]["shapefile_path"])

## Generate ERA5 baseline forcing

- Calls `generate_lumped_ERA5_forcing(...)` for 1940–2015 and saves outputs to the ERA5 folder.

Purpose: produce observational/historical forcing used as baseline.

In [ ]:
generate_lumped_ERA5_forcing(
    start="1940-01-01T00:00:00Z",
    end="2015-12-31T00:00:00Z",
    shapefile=shapefile,
    output_root=ARAL_FORCING_ERA5_FOLDER,
    output_name=shapefile.stem,
)



## Generate CMIP6 historical forcings (per model)

- Loops over `list_models` and calls `generate_lumped_CMIP_historical_forcing(...)` for 1940–2014.
- Creates model-specific output directories and logs success/failure.

Purpose: produce historical model forcings for each configured CMIP6 model.



In [ ]:
for model_name in list_models:
    
    output_path = (
        ARAL_FORCING_CMIP6_HIST_FOLDER / model_name
    )

    try:
        generate_lumped_CMIP_historical_forcing(
            start="1940-01-01T00:00:00Z",
            end="2014-12-31T00:00:00Z",
            shapefile=shapefile,
            output_root=output_path,
            output_name=shapefile.stem,
            model=model_name,
        )
        print(f"Finished {model_name}")

    except Exception as e:
        print(f"Failed {model_name}: {e}")


# Generate CMIP6 future forcings (per model, SSP)

- Loops over `list_models` and calls `generate_lumped_CMIP_future_forcing(...)` for 2015–2099 with `ssp='ssp245'`.
- Writes model outputs and logs progress or errors.

Purpose: produce future scenario forcings for each model.

Note: only generates `SSP2-4.5` here, but similar logic can be applied to make other scenario's

In [ ]:
for model_name in list_models:
    output_path = (
        ARAL_FORCING_CMIP6_FUTURE_FOLDER
        / model_name
    )

    try:
        generate_lumped_CMIP_future_forcing(
            start="2015-01-01T00:00:00Z",
            end="2099-12-31T00:00:00Z",
            shapefile=shapefile,
            output_root=output_path,
            output_name=shapefile.stem,
            model=model_name,
            ssp='ssp245'
        )
        print(f"Finished {model_name}")

    except Exception as e:
        print(f"Failed {model_name}: {e}")